<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/19_end_to_end_rag_pipeline/end_to_end_rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U sentence-transformers transformers scikit-learn faiss-cpu sentencepiece --quiet

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
documents = [
    "Paris is the capital of France.",
    "France is located in Europe.",
    "Berlin is the capital of Germany.",
    "Python is a programming language."
]

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

embeddings = embed_model.encode(documents)

In [ ]:
query = "capital of france"

In [27]:
rewrite_prompt = f"""
Rewrite the following query into a clear and complete QUESTION.
Do not add extra information.
Query: {query}

Output only the question.
"""

inputs = tokenizer(rewrite_prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=30)

query = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("REWRITTEN QUERY:", query)

REWRITTEN QUERY: What is the capital of France?


In [28]:
query_embedding = embed_model.encode([query])
query_tfidf = vectorizer.transform([query])

semantic_scores = cosine_similarity(query_embedding, embeddings)[0]
keyword_scores = cosine_similarity(query_tfidf, tfidf_matrix)[0]

alpha = 0.5
hybrid_scores = alpha * semantic_scores + (1 - alpha) * keyword_scores

In [29]:
top_k = 2

top_indices = hybrid_scores.argsort()[-top_k:][::-1]

context = " ".join([documents[i] for i in top_indices])

confidence = hybrid_scores[top_indices[0]] - hybrid_scores[top_indices[1]]

print("CONFIDENCE:", confidence)
print("CONTEXT:", context)

CONFIDENCE: 0.3384123383419394
CONTEXT: Paris is the capital of France. Berlin is the capital of Germany.


In [30]:
threshold = 0.1

if confidence < threshold:
    print("\nLOW CONFIDENCE → I don't know")
else:
    prompt = f"""
    Use the following context to answer the question.

    Context: {context}

    Question: {query}
    """

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=40)

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("\nFINAL ANSWER:")
    print(answer)


FINAL ANSWER:
Paris
